# PyStars Tutorial

PyStars automates significance testing for biological and life sciences data.
Given a pandas DataFrame with your measurements, it walks a decision flowchart
(normality → equal variance → test selection) so you don't have to.

```
%%uv
pystars
```

In [ ]:
import pystars as ps
import pandas as pd
import numpy as np

---

## 1. Creating example data

Let's create three synthetic datasets we'll use throughout:

In [ ]:
rng = np.random.default_rng(42)

# Two-group independent with a small effect
n = 30
df_2group = pd.DataFrame({
    "group":   ["control"] * n + ["treatment"] * n,
    "value":   np.r_[rng.normal(10, 2, n), rng.normal(11.5, 2, n)],
})

# Three-group with unequal variance (Welch ANOVA scenario)
df_3group = pd.DataFrame({
    "group":   ["A"] * 25 + ["B"] * 25 + ["C"] * 25,
    "value":   np.r_[rng.normal(10, 1, 25),
                      rng.normal(12, 2, 25),
                      rng.normal(14, 3, 25)],
})

# Paired / two-factor data
subjects = [f"mouse_{i}" for i in range(15)]
df_paired = pd.DataFrame({
    "subject":  subjects * 2,
    "group":    ["pre"] * 15 + ["post"] * 15,
    "value":    np.r_[rng.normal(10, 1.5, 15), rng.normal(8, 1.5, 15)],
})

---

## 2. The auto-dispatcher: `pystars.test()`

The main entry point. Pass a DataFrame, tell it which column holds the
measurement (`value`) and which holds the group labels (`group`). PyStars
checks normality & equal variance and picks the right test automatically.

In [ ]:
result = ps.test(df_2group, value="value", group="group")
result.show()

In [ ]:
# Tidy one-row DataFrame export
result.to_dataframe()

The dispatcher also works with more than two groups and with multi-factor designs.

In [ ]:
# Three groups → Kruskal-Wallis or ANOVA depending on normality
result = ps.test(df_3group, value="value", group="group")
result.show()

### Paired test with the dispatcher

For paired designs, pass `subject` and `paired=True`:

In [ ]:
result = ps.test(df_paired, value="value", group="group",
                 subject="subject", paired=True)
result.show()

In [ ]:
# Plain-text summary (good for logging)
print(result.summary())

### Controlling the significance level

The `alpha` parameter controls the threshold for assumption checks and
post-hoc gating (default 0.05):

In [ ]:
ps.test(df_2group, value="value", group="group", alpha=0.01).show()

### Suppressing post-hoc tests

Set `auto_posthoc=False` to skip post-hoc tests after a significant ANOVA:

In [ ]:
ps.test(df_3group, value="value", group="group", auto_posthoc=False).show()

---

## 3. Direct test functions

Each individual test is available as a standalone function. This bypasses the
flowchart and runs the test directly.

In [ ]:
# Welch's t-test (default, safer for biological data)
r1 = ps.ttest(df_2group, value="value", group="group")

# Student's t-test (assumes equal variance)
r2 = ps.ttest(df_2group, value="value", group="group", welch=False)

# Paired t-test
r3 = ps.ttest(df_paired, value="value", group="group",
              subject="subject", paired=True)

# Mann-Whitney U (non-parametric, two independent groups)
r4 = ps.mannwhitney(df_2group, value="value", group="group")

# Wilcoxon signed-rank (non-parametric, paired)
r5 = ps.wilcoxon(df_paired, value="value", group="group",
                 subject="subject")

# One-way ANOVA
r6 = ps.anova(df_3group, value="value", group="group")

# Welch's ANOVA (unequal variances)
r7 = ps.anova(df_3group, value="value", group="group", welch=True)

# Kruskal-Wallis (non-parametric, any number of groups)
r8 = ps.kruskal(df_3group, value="value", group="group")

In [ ]:
# Compare them side by side
ps.to_dataframe([r1, r2, r3, r4, r5, r6, r7, r8])

---

## 4. Assumption checks

You can check normality and equal variance independently:

In [ ]:
# Normality (Shapiro-Wilk) — per group
norm = ps.check_normality(df_2group, value="value", group="group")
norm.show()

# The per-group details are in .details
norm.details

In [ ]:
# Equal variance (Levene's test, median-centred)
var = ps.check_equal_variance(df_2group, value="value", group="group")
var.show()

In [ ]:
# Normality of paired differences
ps.check_normality(df_paired, value="value", group="group",
                   subject="subject", paired=True).show()

---

## 5. Post-hoc tests

Post-hoc comparisons are done manually here; the dispatcher runs them
automatically when `auto_posthoc=True`.

In [ ]:
# Tukey HSD (after one-way ANOVA with equal variance)
ph = ps.posthoc_tukey(df_3group, value="value", group="group")
ph.show()
ph.pairwise

In [ ]:
# Games-Howell (after Welch's ANOVA, unequal variance)
ph = ps.posthoc_games_howell(df_3group, value="value", group="group")
ph.pairwise

In [ ]:
# Dunn's test (after Kruskal-Wallis, non-parametric)
# Default correction: Holm-Bonferroni
ph = ps.posthoc_dunn(df_3group, value="value", group="group")
ph.pairwise

In [ ]:
# Unadjusted Dunn's
ps.posthoc_dunn(df_3group, value="value", group="group",
                p_adjust=None).pairwise

---

## 6. Two-way (factorial) ANOVA

When you have two or more factors, pass a list of column names as `group`:

In [ ]:
df_twoway = pd.DataFrame({
    "genotype":  np.repeat(["WT", "KO"], 30),
    "treatment": np.tile(np.repeat(["saline", "drug"], 15), 2),
    "value":     rng.normal(10, 2, 60),
})
# Inject an interaction effect
df_twoway.loc[
    (df_twoway["genotype"] == "KO") & (df_twoway["treatment"] == "drug"),
    "value"
] += 5

result = ps.anova_twoway(df_twoway, value="value", group=["genotype", "treatment"])
result.show()

In [ ]:
# Full ANOVA table
result.details

The dispatcher also routes multi-factor designs automatically:

In [ ]:
result = ps.test(df_twoway, value="value", group=["genotype", "treatment"])
result.show()

---

## 7. Wide format data

PyStars accepts wide format where each group has its own column:

In [ ]:
df_wide = pd.DataFrame({
    "mouse": ["m1", "m2", "m3", "m4", "m5"],
    "control": rng.normal(10, 2, 5),
    "drug_A": rng.normal(12, 2, 5),
    "drug_B": rng.normal(9, 2, 5),
})

result = ps.test(df_wide, format="wide",
                 groups=["control", "drug_A", "drug_B"],
                 subject_index="mouse")
result.show()

In [ ]:
# Direct test with wide format
ps.kruskal(df_wide, format="wide", groups=["control", "drug_A", "drug_B"]).show()

---

## 8. Working with results

Every function returns a `TestResult` object with several useful methods.

In [ ]:
result = ps.test(df_2group, value="value", group="group")

# Inspect fields
print(f"Test:       {result.test_name}")
print(f"Statistic:  {result.statistic:.3f}")
print(f"p-value:    {result.p_value:.4f}")
print(f"Effect size: {result.effect_size}")

In [ ]:
# Tidy DataFrame for export / programmatic use
result.to_dataframe()

In [ ]:
# Combining multiple results
r_a = ps.ttest(df_2group, value="value", group="group")
r_b = ps.anova(df_3group, value="value", group="group")
r_c = ps.kruskal(df_3group, value="value", group="group")

ps.to_dataframe([r_a, r_b, r_c])

---

## 9. The decision flowchart at a glance

The dispatcher implements this logic:

| Data shape | Assumptions | Selected test |
|---|---|---|
| 2 groups, independent | normal + equal var | Student's t-test |
| 2 groups, independent | normal + unequal var | Welch's t-test |
| 2 groups, independent | non-normal / small n | Mann-Whitney U |
| 2 groups, paired | normal differences | Paired t-test |
| 2 groups, paired | non-normal differences | Wilcoxon signed-rank |
| >2 groups, one factor | normal + equal var | One-way ANOVA + Tukey HSD |
| >2 groups, one factor | normal + unequal var | Welch's ANOVA + Games-Howell |
| >2 groups, one factor | non-normal | Kruskal-Wallis + Dunn's test |
| >=2 factors | — | Two-way ANOVA (interaction reported) |

Post-hoc tests run only when the omnibus test is significant
(p < `alpha`) and `auto_posthoc=True`. Groups with fewer than 3
observations are treated as non-normal.